# Four logo designs: Scheffé's paired comparison

**Paired Comparison** · a worked example from the StatNex teaching guide

**The story.** A start-up shortlisted four logo designs. For each pair of logos, shown in each order, a different group of twelve people scored from -2 to +2 how much they preferred the logo shown first.

**The question.** Which logo do people prefer, and are the differences real?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/paired_comparison.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
First logo,Second logo,-2,-1,0,1,2
Logo A,Logo B,3.0,6.0,3.0,0.0,0.0
Logo B,Logo A,0.0,0.0,1.0,6.0,5.0
Logo A,Logo C,2.0,6.0,4.0,0.0,0.0
Logo C,Logo A,0.0,0.0,6.0,5.0,1.0
Logo A,Logo D,8.0,4.0,0.0,0.0,0.0
Logo D,Logo A,0.0,0.0,0.0,1.0,11.0
Logo B,Logo C,0.0,1.0,1.0,9.0,1.0
Logo C,Logo B,1.0,5.0,3.0,2.0,1.0
Logo B,Logo D,1.0,9.0,1.0,1.0,0.0
Logo D,Logo B,0.0,0.0,3.0,5.0,4.0
Logo C,Logo D,3.0,4.0,5.0,0.0,0.0
Logo D,Logo C,0.0,0.0,1.0,7.0,4.0

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per pair as it was presented: the sample shown first, the sample shown second,
# and how many judges gave each score. The score is the column header (-2 ... 2), and a
# positive score favours the sample shown first.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
first, second = "First logo", "Second logo"
score_columns = ["-2", "-1", "0", "1", "2"]
points = np.array([float(column) for column in score_columns])
counts = data[score_columns].fillna(0).to_numpy(float)

# The samples, in the order they first appear (the first-shown column, then the other).
samples = list(dict.fromkeys([*data[first].astype(str), *data[second].astype(str)]))
t = len(samples)
index = {name: i for i, name in enumerate(samples)}

# totals[i, j] is the sum of all scores when sample i was shown first and j second.
totals = np.zeros((t, t))
for row in range(len(data)):
    i, j = index[str(data[first][row])], index[str(data[second][row])]
    totals[i, j] += counts[row] @ points
observations = counts.sum()
# n is how many judgements each pair received, both presentation orders together.
n = observations / (t * (t - 1) / 2)

## Analysis of variance table

In [ ]:
# The mean preference alpha is how much a sample is preferred over all the others on
# average. The combination effect is what is left of each pair once the two alphas
# are taken away; the order effect is the part of a pair that does not cancel when
# it is shown both ways round. The total is the uncorrected sum of squared scores.
alpha = (totals - totals.T).sum(axis=1) / (n * t)
ss_main = n * t * (alpha**2).sum()
ss_combination = ss_order = 0.0
for i, j in itertools.combinations(range(t), 2):
    gamma = (totals[i, j] - totals[j, i]) / n - (alpha[i] - alpha[j])
    delta = (totals[i, j] + totals[j, i]) / n
    ss_combination += n * gamma**2
    ss_order += n * delta**2
ss_total = (counts @ points**2).sum()
effects = {"Main effect": (ss_main, t - 1),
           "Combination effect": (ss_combination, (t - 1) * (t - 2) // 2)}
# Scheffé's original method (and Ura's variant) separate the order effect.
effects["Order effect"] = (ss_order, t * (t - 1) // 2)
ss_residual = ss_total - sum(ss for ss, df in effects.values())
df_residual = observations - sum(df for ss, df in effects.values())
ms_residual = ss_residual / df_residual
rows = [[ss, df, ss / df, ss / df / ms_residual,
         stats.f.sf(ss / df / ms_residual, df, df_residual)] for ss, df in effects.values()]
rows.append([ss_residual, df_residual, ms_residual, np.nan, np.nan])
rows.append([ss_total, observations, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, index=[*effects, "Residual", "Total"],
                     columns=["SS", "df", "MS", "F", "P"])
print(anova)

## Mean preference and the yardstick

In [ ]:
# The yardstick is a studentised-range interval, so it holds for every pair at once:
# two samples further apart than it differ.
level = 95 / 100
q = stats.studentized_range.ppf(level, t, df_residual)
yardstick = q * np.sqrt(ms_residual / (n * t))
print(f"\nyardstick = {yardstick:.6f}  (q = {q:.6f})")
means = pd.DataFrame({"alpha": alpha, "lower": alpha - yardstick, "upper": alpha + yardstick},
                     index=samples)
print(means)

rows = []
for i, j in itertools.combinations(range(t), 2):
    difference = alpha[i] - alpha[j]
    rows.append([samples[i], samples[j], difference, difference - yardstick,
                 difference + yardstick])
differences = pd.DataFrame(rows, columns=["sample 1", "sample 2", "difference", "lower",
                                          "upper"])
print()
print(differences.to_string(index=False))

## What the result says

People judged the logos two at a time, and the method turns the scores into a mean preference for each logo. In the analysis of variance table, the main effect has F = 111.1 and a P value very close to 0. F compares the differences between logos with the noise, so the logos are not equally liked.

Logo D has the highest mean preference, 0.95, and Logo A the lowest, -0.92. A confidence interval is the range of values that fit the data. For every pair of logos, the interval of the difference leaves out 0. Even the closest pair, B and C, differs by 0.41, with an interval from 0.13 to 0.68.

Choose Logo D, but remember these are test judges, not customers in daily life.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.